# Qwen3 Checkpoint-250 Test Inference Only

Loads only this adapter and writes one submission file:

`/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1/runs/20260716_065651/qwen3vl_4b_4task/checkpoint-250`

No Qwen2 baseline, no validation, no tuning, no holdout.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path('/content/.snu_qwen3_checkpoint250_test_deps_installed')

if not MARKER.exists():
    packages = [
        'transformers>=4.57.0',
        'accelerate>=0.34.0',
        'bitsandbytes>=0.46.1',
        'peft',
        'qwen-vl-utils',
        'modelscope',
        'hf_transfer',
        'jedi',
        'pandas==2.2.2',
        'safetensors>=0.4.5',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])
    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime. Run this cell again after restart.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed. Continue.')

In [ ]:
# 2) Setup
from google.colab import drive
drive.mount('/content/drive')

import gc
import itertools
import json
import math
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
import random
import re
import shutil
import subprocess
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, set_seed
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    Qwen3VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import PeftModel
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

ZIP_PATH = '/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip'
DATA_DIR = '/content/snuaichallenge_data'
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TEST_IMAGE_DIR = os.path.join(DATA_DIR, 'test')

MODEL_REPO_ID = 'Qwen/Qwen3-VL-4B-Instruct'
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen3-VL-4B-Instruct'
ADAPTER_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1/runs/20260716_065651/qwen3vl_4b_4task/checkpoint-250'
USE_LOCAL_STAGING = True
LOCAL_MODEL_DIR = '/content/qwen3_vl_4b_instruct_local'
LOCAL_ADAPTER_DIR = '/content/qwen3_checkpoint250_adapter_local'

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = '/content/drive/MyDrive/SNU_AI_Challenge/qwen3_checkpoint250_test_only'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, RUN_ID)
CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache')
SUBMIT_PATH = os.path.join(OUTPUT_DIR, 'submission_qwen3_checkpoint250.csv')
for path in [OUTPUT_ROOT, OUTPUT_DIR, CACHE_DIR]:
    os.makedirs(path, exist_ok=True)

SEED = 42
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_BATCH_SIZE = 2
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

# Structured decoding weights. Change these only if you have a known tuned config.
ALPHA = 1.0
BETA = 1.0
GAMMA = 1.0
PAIRWISE_BIDIRECTIONAL = True

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall('/content/')

assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')), ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

test_df = pd.read_csv(TEST_CSV)
test_df['Id'] = test_df['Id'].astype(str)

assert torch.cuda.is_available(), 'GPU runtime is required. In Colab: Runtime > Change runtime type > GPU.'
print('cuda:', torch.cuda.get_device_name(0))

print('adapter:', ADAPTER_DIR)
print('output:', OUTPUT_DIR)
print('test rows:', len(test_df))

In [ ]:
# 3) Helpers
def model_cache_is_complete(model_dir):
    if not os.path.exists(os.path.join(model_dir, 'config.json')):
        return False
    has_weight = any(os.path.exists(os.path.join(model_dir, name)) for name in ['model.safetensors.index.json', 'pytorch_model.bin', 'pytorch_model.bin.index.json']) or bool([name for name in os.listdir(model_dir) if name.endswith('.safetensors')])
    has_processor = any(os.path.exists(os.path.join(model_dir, name)) for name in ['preprocessor_config.json', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json'])
    return bool(has_weight and has_processor)


def adapter_cache_is_complete(adapter_dir):
    return os.path.exists(os.path.join(adapter_dir, 'adapter_config.json')) and any(
        os.path.exists(os.path.join(adapter_dir, name))
        for name in ['adapter_model.safetensors', 'adapter_model.bin']
    )


def copy_dir_to_local(src, dst, complete_fn, label):
    if not USE_LOCAL_STAGING:
        return src
    if complete_fn(dst):
        print(f'Using staged {label}:', dst)
        return dst
    print(f'Staging {label} to local disk:')
    print('  from:', src)
    print('  to  :', dst)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        subprocess.run(['rsync', '-a', '--delete', src.rstrip('/') + '/', dst.rstrip('/') + '/'], check=True)
    except Exception as exc:
        print('rsync failed; falling back to shutil.copytree:', repr(exc))
        shutil.copytree(src, dst, dirs_exist_ok=True)
    assert complete_fn(dst), f'Local staged {label} is incomplete: {dst}'
    return dst


def ensure_base_model_path():
    if model_cache_is_complete(DRIVE_MODEL_DIR):
        print('Using cached base model:', DRIVE_MODEL_DIR)
        return copy_dir_to_local(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, model_cache_is_complete, 'base model')
    if not USE_MODELSCOPE_BASE_MODEL:
        print('Using Hugging Face repo id:', MODEL_REPO_ID)
        return MODEL_REPO_ID
    print('Base model cache not found. Downloading via ModelScope:', MODEL_REPO_ID)
    from modelscope import snapshot_download as modelscope_snapshot_download
    model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir='/content/modelscope_cache')
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        tmp = DRIVE_MODEL_DIR + '.tmp'
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
        shutil.copytree(model_dir, tmp, dirs_exist_ok=True)
        if os.path.exists(DRIVE_MODEL_DIR):
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
    return copy_dir_to_local(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, model_cache_is_complete, 'base model')


def load_model_class():
    if Qwen3VLForConditionalGeneration is not None:
        return Qwen3VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError('No compatible Qwen3-VL model class found. Re-run dependency cell with transformers>=4.57.0 and restart runtime.')


def row_image_paths(row, image_root):
    sample_id = str(row['Id'])
    return [os.path.join(image_root, sample_id, str(row[f'Input_{i}'])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert('RGB').copy()


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def task_instruction(example):
    sentence = example['sentence']
    task_type = example['task_type']
    if task_type == 'pairwise':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Which image occurs first?\n'
            'If the first image occurs earlier, answer 1.\n'
            'If the second image occurs earlier, answer 2.\n'
            'Answer only 1 or 2.'
        )
    if task_type == 'first':
        return f'Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4.'
    if task_type == 'last':
        return f'Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4.'
    raise ValueError(task_type)


def make_messages(example):
    content = []
    for idx, _ in enumerate(example['image_paths'], start=1):
        content.append({'type': 'text', 'text': f'\nImage {idx}:'})
        content.append({'type': 'image'})
    content.append({'type': 'text', 'text': '\n\n' + task_instruction(example)})
    return [{'role': 'user', 'content': content}]


def make_test_example(row, task_type, pair=None):
    image_paths = row_image_paths(row, TEST_IMAGE_DIR)
    example = {
        'sample_id': str(row['Id']),
        'sentence': '' if pd.isna(row['Sentence']) else str(row['Sentence']),
        'image_paths': image_paths,
        'task_type': task_type,
        'target': '1',
    }
    if task_type == 'pairwise':
        a, b = pair
        example['image_paths'] = [image_paths[a - 1], image_paths[b - 1]]
    return example

In [ ]:
# 4) Load Qwen3 checkpoint only
MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
ADAPTER_LOCAL_DIR = copy_dir_to_local(ADAPTER_DIR, LOCAL_ADAPTER_DIR, adapter_cache_is_complete, 'adapter')

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

def digit_token_id(digit):
    ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f'Digit {digit} tokenized to {ids}')
    return ids[0]

DIGIT_TOKEN_IDS = {digit: digit_token_id(digit) for digit in [1, 2, 3, 4]}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = load_model_class().from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_LOCAL_DIR, is_trainable=False)
model.eval()
if hasattr(model, 'generation_config'):
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    model.generation_config.num_beams = 1

print('loaded Qwen3 checkpoint')
print('digit token ids:', DIGIT_TOKEN_IDS)

In [ ]:
# 5) Batched structured inference
def model_device():
    return next(model.parameters()).device


@torch.no_grad()
def score_digit_candidate_batch(examples, candidates_per_example):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = 'right'
        texts = [processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True) for example in examples]
        image_cache = {}

        def cached_load(path):
            if path not in image_cache:
                image_cache[path] = load_rgb(path)
            return image_cache[path]

        images = [[cached_load(path) for path in example['image_paths']] for example in examples]
        inputs = processor(text=texts, images=images, padding=True, return_tensors='pt')
        inputs = {key: value.to(model_device()) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = model(**inputs)
        results = []
        for row_index, candidates in enumerate(candidates_per_example):
            last_pos = int(inputs['attention_mask'][row_index].sum().item()) - 1
            logits = outputs.logits[row_index, last_pos]
            token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
            probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
            results.append({int(candidate): float(prob) for candidate, prob in zip(candidates, probs)})
        return results
    finally:
        processor.tokenizer.padding_side = old_padding_side


def score_test_tasks(row):
    endpoint_probs = score_digit_candidate_batch(
        [make_test_example(row, 'first'), make_test_example(row, 'last')],
        [[1, 2, 3, 4], [1, 2, 3, 4]],
    )
    first_probs = {str(k): v for k, v in endpoint_probs[0].items()}
    last_probs = {str(k): v for k, v in endpoint_probs[1].items()}

    forward_examples, forward_candidates = [], []
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        forward_examples.append(make_test_example(row, 'pairwise', pair=(a, b)))
        forward_candidates.append([1, 2])
    forward_probs = []
    for start in range(0, len(forward_examples), PAIR_BATCH_SIZE):
        forward_probs.extend(score_digit_candidate_batch(
            forward_examples[start:start + PAIR_BATCH_SIZE],
            forward_candidates[start:start + PAIR_BATCH_SIZE],
        ))

    reverse_probs = None
    if PAIRWISE_BIDIRECTIONAL:
        reverse_examples, reverse_candidates = [], []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            reverse_examples.append(make_test_example(row, 'pairwise', pair=(b, a)))
            reverse_candidates.append([1, 2])
        reverse_probs = []
        for start in range(0, len(reverse_examples), PAIR_BATCH_SIZE):
            reverse_probs.extend(score_digit_candidate_batch(
                reverse_examples[start:start + PAIR_BATCH_SIZE],
                reverse_candidates[start:start + PAIR_BATCH_SIZE],
            ))

    pair_probs = {}
    for idx, (first_index, second_index) in enumerate(PAIR_INDICES):
        a, b = first_index + 1, second_index + 1
        p_a_before_b = float(forward_probs[idx][1])
        if PAIRWISE_BIDIRECTIONAL:
            p_a_before_b = 0.5 * (p_a_before_b + float(reverse_probs[idx][2]))
        pair_probs[f'{a}>{b}'] = float(p_a_before_b)
        pair_probs[f'{b}>{a}'] = float(1.0 - p_a_before_b)

    return first_probs, last_probs, pair_probs


def structured_score(sample, order, alpha=ALPHA, beta=BETA, gamma=GAMMA):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(sample['pair_probs'][f'{order[i]}>{order[j]}']) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(sample['first_probs'][str(order[0])]) + eps)
    last_score = math.log(float(sample['last_probs'][str(order[-1])]) + eps)
    return float(alpha * pair_score + beta * first_score + gamma * last_score)


def decode_structured(sample):
    return list(max(PERMUTATIONS, key=lambda order: structured_score(sample, order)))

In [ ]:
# 6) Run test inference and save submission
submission_rows = []
cache_path = os.path.join(CACHE_DIR, 'test_probability_cache.jsonl')
if os.path.exists(cache_path):
    os.remove(cache_path)

with open(cache_path, 'a', encoding='utf-8') as cache_file:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='qwen3 checkpoint-250 test'):
        first_probs, last_probs, pair_probs = score_test_tasks(row)
        sample = {
            'sample_id': str(row['Id']),
            'first_probs': first_probs,
            'last_probs': last_probs,
            'pair_probs': pair_probs,
        }
        pred_order = decode_structured(sample)
        sample['pred_order'] = pred_order
        submission_rows.append({'Id': str(row['Id']), 'Answer': str(sequence_to_answer(pred_order))})
        cache_file.write(json.dumps(sample, ensure_ascii=False) + '\n')
        cache_file.flush()

submission = pd.DataFrame(submission_rows)
submission.to_csv(SUBMIT_PATH, index=False)

run_config = {
    'adapter_dir': ADAPTER_DIR,
    'adapter_local_dir': ADAPTER_LOCAL_DIR,
    'model_repo_id': MODEL_REPO_ID,
    'model_id': MODEL_ID,
    'use_local_staging': USE_LOCAL_STAGING,
    'alpha': ALPHA,
    'beta': BETA,
    'gamma': GAMMA,
    'pairwise_bidirectional': PAIRWISE_BIDIRECTIONAL,
    'pair_batch_size': PAIR_BATCH_SIZE,
    'min_pixels': MIN_PIXELS,
    'max_pixels': MAX_PIXELS,
    'test_rows': len(test_df),
    'submission_path': SUBMIT_PATH,
    'cache_path': cache_path,
}
with open(os.path.join(OUTPUT_DIR, 'run_config.json'), 'w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

display(submission.head())
print('submission saved:', SUBMIT_PATH)
print('cache saved:', cache_path)